# **FRAMEWORK V7 — C17 GOBIERNO DE MODELOS**

Este notebook implementa la capa de **gobierno, trazabilidad, control de uso y documentación** de los modelos desarrollados en Framework V7.

La versión actual gobierna conjuntamente los experimentos **Exp01** y **Exp04** y consume la evidencia producida por C13, C14, C15 y, cuando ya se encuentre publicada, C16.

> **Principio de gobierno:** C17 no vuelve a entrenar ni recalcula predicciones. Su función es verificar evidencia, registrar versiones, establecer restricciones de uso, documentar riesgos y decidir si un modelo puede avanzar a un piloto controlado o debe permanecer bloqueado para uso operacional.


## **M0. Configuración del Gobierno**

La configuración del portafolio se concentra en una sola celda. El notebook procesa ambos experimentos en una ejecución y conserva la separación entre objetivo científico y objetivo operativo del modelo.


In [4]:
#==========================================================================================
# SCRIPT 90
# CONFIGURACIÓN DEL GOBIERNO DE MODELOS
#==========================================================================================

EXPERIMENTOS = ["Exp01", "Exp04"]

REPO = "jriatiga/FRAMEWORK_V7"
RAMA = "main"
RAW_BASE = f"https://raw.githubusercontent.com/{REPO}/refs/heads/{RAMA}/"
API_COMMIT = f"https://api.github.com/repos/{REPO}/commits/{RAMA}"

# C16 se considera fuente opcional porque puede ejecutarse y publicarse después de C15.
BASE_C16_REPO = "DATA/INTERPRETACION"
USAR_C16_SI_DISPONIBLE = True

CARPETA_SALIDA = "gobierno_modelos"

print()
print("=" * 90)
print("FRAMEWORK V7 - CONFIGURACIÓN C17")
print("=" * 90)
print()
print(f"Repositorio        : {REPO}")
print(f"Rama               : {RAMA}")
print(f"Experimentos       : {EXPERIMENTOS}")
print(f"Salida local       : {CARPETA_SALIDA}")
print(f"C16 opcional       : {USAR_C16_SI_DISPONIBLE}")



FRAMEWORK V7 - CONFIGURACIÓN C17

Repositorio        : jriatiga/FRAMEWORK_V7
Rama               : main
Experimentos       : ['Exp01', 'Exp04']
Salida local       : gobierno_modelos
C16 opcional       : True


## **M1. Librerías y utilidades**

Se utilizan funciones tolerantes a acentos y variantes de nombres de parámetros para evitar que diferencias menores de escritura rompan la trazabilidad. Las descargas obligatorias detienen la ejecución si faltan; las evidencias de C16 son opcionales hasta que hayan sido publicadas.


In [7]:
import os
import io
import json
import hashlib
import zipfile
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd
import requests

os.makedirs(CARPETA_SALIDA, exist_ok=True)


def normalizar_texto(valor):
    texto = str(valor).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.replace("_", " ").split())


def obtener_parametro(df, nombres, default=np.nan, requerido=False):
    if df is None or len(df) == 0:
        if requerido:
            raise KeyError(f"No existe metadata para recuperar: {nombres}")
        return default

    if isinstance(nombres, str):
        nombres = [nombres]

    if not {"Parametro", "Valor"}.issubset(df.columns):
        if requerido:
            raise KeyError("La tabla no contiene columnas Parametro/Valor.")
        return default

    mapa = {
        normalizar_texto(parametro): valor
        for parametro, valor in zip(df["Parametro"], df["Valor"])
    }

    for nombre in nombres:
        clave = normalizar_texto(nombre)
        if clave in mapa:
            return mapa[clave]

    if requerido:
        raise KeyError(f"Parámetro no encontrado: {nombres}")
    return default


def a_bool(valor, default=False):
    if pd.isna(valor):
        return default
    if isinstance(valor, (bool, np.bool_)):
        return bool(valor)
    texto = normalizar_texto(valor)
    if texto in {"true", "verdadero", "1", "si", "yes"}:
        return True
    if texto in {"false", "falso", "0", "no"}:
        return False
    return default


def leer_data_url(url, obligatorio=True, timeout=45):
    try:
        respuesta = requests.get(
            url,
            timeout=timeout,
            headers={"Cache-Control": "no-cache"}
        )
        respuesta.raise_for_status()
        if url.endswith(".csv"):
            return pd.read_csv(io.BytesIO(respuesta.content), encoding="utf-8-sig")
        elif url.endswith(".xlsx"):
            return pd.read_excel(io.BytesIO(respuesta.content))
        else:
            if obligatorio:
                raise ValueError(f"Formato de archivo no soportado para URL obligatoria: {url}")
            return None
    except Exception as exc:
        if obligatorio:
            raise RuntimeError(f"No fue posible cargar artefacto obligatorio:\n{url}\n{exc}") from exc
        return None


def valor_numero(valor, default=np.nan):
    try:
        if pd.isna(valor):
            return default
        return float(str(valor).replace(",", "."))
    except Exception:
        return default


def sha256_archivo(ruta, bloque=1024 * 1024):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        while True:
            b = f.read(bloque)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print("Utilidades cargadas correctamente.")

Utilidades cargadas correctamente.


## **M2. Fuentes oficiales del ciclo de vida**

C17 registra como fuentes principales:

- **C13:** preparación de Machine Learning y secuencias sin escalamiento ajustado;
- **C14:** split temporal por Nodo, escalamiento ajustado solo con entrenamiento, entrenamiento y diagnóstico;
- **C15:** reproducción independiente de inferencia y evaluación oficial sobre la partición de prueba;
- **C16:** interpretación de resultados, cuando sus artefactos ya estén publicados.

La ausencia de C16 no invalida C17, pero queda registrada como evidencia pendiente de publicación.


In [8]:
#==========================================================================================
# SCRIPT 92
# CONSTRUCCIÓN Y CARGA DE FUENTES
#==========================================================================================

FUENTES = {}

for exp in EXPERIMENTOS:
    rutas = {
        # C13
        "c13_registro": (
            f"{RAW_BASE}DATA/MACHINE_LEARNING/C13_MACHINE_LEARNING/"
            f"Transformaciones/{exp}/registro_preparacion.csv"
        ),

        # C14
        "c14_metadata_modelo": f"{RAW_BASE}DATA/MODELADO/Modelos/{exp}/metadata_modelo.csv",
        "c14_registro": f"{RAW_BASE}DATA/MODELADO/Metricas/{exp}/registro_{exp}.csv",
        "c14_diagnostico": f"{RAW_BASE}DATA/MODELADO/Diagnosticos/{exp}/diagnostico_modelo_{exp}.csv",
        "c14_recomendaciones": f"{RAW_BASE}DATA/MODELADO/Diagnosticos/{exp}/recomendaciones_{exp}.csv",
        "c14_manifiesto": f"{RAW_BASE}DATA/MODELADO/Metricas/{exp}/manifiesto_artefactos_{exp}.csv",

        # C15
        "c15_metadata": f"{RAW_BASE}DATA/EVALUACIONES/{exp}/metadata_prediccion.xlsx",
        "c15_metricas": f"{RAW_BASE}DATA/EVALUACIONES/{exp}/metricas_particiones.csv",
        "c15_metricas_nodo": f"{RAW_BASE}DATA/EVALUACIONES/{exp}/metricas_por_nodo.csv",
        "c15_integridad": f"{RAW_BASE}DATA/EVALUACIONES/{exp}/validacion_integridad.csv",
        "c15_manifiesto": f"{RAW_BASE}DATA/EVALUACIONES/{exp}/manifiesto_artefactos_C15_{exp}.csv",
        "c15_baselines": f"{RAW_BASE}DATA/EVALUACIONES/{exp}/baselines.csv",

        # C16 - opcional
        "c16_metadata": f"{RAW_BASE}{BASE_C16_REPO}/{exp}/metadata_interpretacion.csv",
        "c16_validacion": f"{RAW_BASE}{BASE_C16_REPO}/{exp}/validacion_final_C16.csv",
        "c16_hallazgos": f"{RAW_BASE}{BASE_C16_REPO}/{exp}/hallazgos.csv",
        "c16_recomendaciones": f"{RAW_BASE}{BASE_C16_REPO}/{exp}/recomendaciones_C16.csv",
    }

    tablas = {}
    for nombre, url in rutas.items():
        obligatorio = not nombre.startswith("c16_")
        if nombre == "c15_baselines" and exp != "Exp04":
            obligatorio = False
        tablas[nombre] = leer_data_url(url, obligatorio=obligatorio)

    FUENTES[exp] = {
        "rutas": rutas,
        "tablas": tablas
    }

# Commit de referencia del repositorio al momento de gobernar.
try:
    resp_commit = requests.get(API_COMMIT, timeout=30)
    resp_commit.raise_for_status()
    info_commit = resp_commit.json()
    COMMIT_SHA = info_commit.get("sha", "NO_DISPONIBLE")
    COMMIT_FECHA = (
        info_commit.get("commit", {})
        .get("committer", {})
        .get("date", "NO_DISPONIBLE")
    )
except Exception:
    COMMIT_SHA = "NO_DISPONIBLE"
    COMMIT_FECHA = "NO_DISPONIBLE"

print()
print("=" * 90)
print("FUENTES CARGADAS")
print("=" * 90)
print()
print(f"Commit referencia : {COMMIT_SHA}")
print(f"Fecha commit       : {COMMIT_FECHA}")
print()

for exp in EXPERIMENTOS:
    print(f"{exp}:")
    for nombre, tabla in FUENTES[exp]["tablas"].items():
        estado = "OK" if tabla is not None else "NO DISPONIBLE / OPCIONAL"
        print(f"  - {nombre:<24}: {estado}")


FUENTES CARGADAS

Commit referencia : ce4ef76511f43f48eb47c637b6ad10db29cf414d
Fecha commit       : 2026-08-12T22:15:46Z

Exp01:
  - c13_registro            : OK
  - c14_metadata_modelo     : OK
  - c14_registro            : OK
  - c14_diagnostico         : OK
  - c14_recomendaciones     : OK
  - c14_manifiesto          : OK
  - c15_metadata            : OK
  - c15_metricas            : OK
  - c15_metricas_nodo       : OK
  - c15_integridad          : OK
  - c15_manifiesto          : OK
  - c15_baselines           : NO DISPONIBLE / OPCIONAL
  - c16_metadata            : NO DISPONIBLE / OPCIONAL
  - c16_validacion          : NO DISPONIBLE / OPCIONAL
  - c16_hallazgos           : NO DISPONIBLE / OPCIONAL
  - c16_recomendaciones     : NO DISPONIBLE / OPCIONAL
Exp04:
  - c13_registro            : OK
  - c14_metadata_modelo     : OK
  - c14_registro            : OK
  - c14_diagnostico         : OK
  - c14_recomendaciones     : OK
  - c14_manifiesto          : OK
  - c15_metadata           

## **M3. Validación de trazabilidad C13 → C15/C16**

Antes de emitir una decisión de gobierno se verifica que los artefactos pertenezcan al mismo experimento y que C15 reporte integridad de entrada. Una inconsistencia de trazabilidad **bloquea automáticamente** cualquier promoción del modelo.


In [11]:
#==========================================================================================
# SCRIPT 93
# VALIDACIÓN DE TRAZABILIDAD
#==========================================================================================

filas_validacion = []

for exp in EXPERIMENTOS:
    t = FUENTES[exp]["tablas"]

    c13_exp = obtener_parametro(t["c13_registro"], "Experimento", default="")
    c14_exp = (
        str(t["c14_registro"].iloc[0].get("Experimento", ""))
        if t["c14_registro"] is not None and len(t["c14_registro"]) > 0
        else ""
    )
    c15_exp = obtener_parametro(t["c15_metadata"], "Experimento", default="")
    c16_exp = obtener_parametro(t["c16_metadata"], "Experimento", default="")

    integridad_c15 = True
    if t["c15_integridad"] is not None and len(t["c15_integridad"]) > 0:
        if {"Control", "Estado"}.issubset(t["c15_integridad"].columns):
            integridad_c15 = all(
                a_bool(v, default=False)
                for v in t["c15_integridad"]["Estado"]
            )

    estado_c14 = obtener_parametro(
        t["c15_metadata"], "Estado C14", default="NO DISPONIBLE"
    )
    estado_c15 = obtener_parametro(
        t["c15_metadata"], "Estado C15", default="NO DISPONIBLE"
    )
    estados_coinciden = a_bool(
        obtener_parametro(t["c15_metadata"], "Estados Coinciden", default=False),
        default=False
    )

    controles = {
        "C13_corresponde_experimento": normalizar_texto(c13_exp) == normalizar_texto(exp),
        "C14_corresponde_experimento": normalizar_texto(c14_exp) == normalizar_texto(exp),
        "C15_corresponde_experimento": normalizar_texto(c15_exp) == normalizar_texto(exp),
        "C15_integridad_completa": bool(integridad_c15),
        "C14_C15_estados_coinciden": bool(estados_coinciden),
        "C16_corresponde_si_disponible": (
            True if not c16_exp else normalizar_texto(c16_exp) == normalizar_texto(exp)
        )
    }

    for control, estado in controles.items():
        filas_validacion.append({
            "Experimento": exp,
            "Control": control,
            "Estado": bool(estado),
            "Detalle": (
                f"Estado C14={estado_c14}; Estado C15={estado_c15}"
                if control == "C14_C15_estados_coinciden"
                else ""
            )
        })

validacion_trazabilidad = pd.DataFrame(filas_validacion)

print()
print("=" * 90)
print("VALIDACIÓN DE TRAZABILIDAD")
print("=" * 90)
print()
display(validacion_trazabilidad)


VALIDACIÓN DE TRAZABILIDAD



,Experimento,Control,Estado,Detalle
0,Exp01,C13_corresponde_experimento,True,
1,Exp01,C14_corresponde_experimento,True,
2,Exp01,C15_corresponde_experimento,True,
3,Exp01,C15_integridad_completa,True,
4,Exp01,C14_C15_estados_coinciden,True,Estado C14=EVALUACION LIMITADA - TEST CON UNA ...
5,Exp01,C16_corresponde_si_disponible,True,
6,Exp04,C13_corresponde_experimento,True,
7,Exp04,C14_corresponde_experimento,True,
8,Exp04,C15_corresponde_experimento,True,
9,Exp04,C15_integridad_completa,True,


## **M4. Identificación y versionamiento**

El identificador gobernado combina:

1. experimento;
2. objetivo operativo y científico;
3. arquitectura;
4. ventana/horizonte;
5. transformación;
6. commit del repositorio al momento de C17;
7. hashes disponibles en los manifiestos de C14/C15.

No se utiliza ya `Exp01-V3`: el modelo vigente del flujo corregido es **Exp01**, cuyo objetivo operativo es `nivel_de_riesgo` y cuyo objetivo científico es `irca`.


In [12]:
#==========================================================================================
# SCRIPT 94
# CATÁLOGO DE MODELOS GOBERNADOS
#==========================================================================================

filas_catalogo = []

for exp in EXPERIMENTOS:
    t = FUENTES[exp]["tablas"]
    m15 = t["c15_metadata"]
    r14 = t["c14_registro"]

    fila14 = r14.iloc[0].to_dict() if r14 is not None and len(r14) else {}

    tipo = str(obtener_parametro(m15, "Tipo Problema", default=fila14.get("Tipo_Problema", "")))
    objetivo = str(obtener_parametro(m15, "Variable Objetivo", default=fila14.get("Variable_Objetivo", "")))
    objetivo_cientifico = str(obtener_parametro(
        m15,
        ["Variable Objetivo Cientifico", "Variable Objetivo Científico"],
        default=fila14.get("Variable_Objetivo_Cientifico", objetivo)
    ))
    modelo = str(obtener_parametro(m15, "Modelo", default="LSTM"))
    dominio = str(obtener_parametro(m15, "Dominio", default=fila14.get("Dominio", "")))
    ventana = int(valor_numero(obtener_parametro(m15, "Ventana", default=12), 12))
    horizonte = int(valor_numero(obtener_parametro(m15, "Horizonte", default=1), 1))
    n_vars = int(valor_numero(obtener_parametro(m15, "Variables Predictoras", default=np.nan), 0))
    variables = str(obtener_parametro(m15, "Variables Modelo", default=""))
    transformacion_x = str(obtener_parametro(m15, "Metodo Transformacion", default="Ninguno"))
    scaler_x = str(obtener_parametro(m15, "Scaler X", default="No aplica"))
    transformacion_y = str(obtener_parametro(m15, "Transformacion Objetivo", default="No aplica"))
    scaler_y = str(obtener_parametro(m15, "Scaler y", default="No aplica"))

    filas_catalogo.append({
        "ID_Modelo_Gobernado": f"{exp}@{COMMIT_SHA[:12] if COMMIT_SHA != 'NO_DISPONIBLE' else 'sin_sha'}",
        "Experimento": exp,
        "Dominio": dominio,
        "Tipo_Problema": tipo,
        "Variable_Objetivo_Modelo": objetivo,
        "Variable_Objetivo_Cientifico": objetivo_cientifico,
        "Artefacto_Modelo": modelo,
        "Ventana": ventana,
        "Horizonte": horizonte,
        "Variables_Predictoras": n_vars,
        "Lista_Predictoras": variables,
        "Transformacion_X": transformacion_x,
        "Scaler_X": scaler_x,
        "Ajuste_Scaler_X": "C14 - solo entrenamiento",
        "Transformacion_y": transformacion_y,
        "Scaler_y": scaler_y,
        "Ajuste_Scaler_y": (
            "C14 - solo y_train" if normalizar_texto(tipo).startswith("reg") else "No aplica"
        ),
        "Muestras_Train": int(valor_numero(obtener_parametro(m15, "Muestras Train", default=0), 0)),
        "Muestras_Validacion": int(valor_numero(obtener_parametro(m15, "Muestras Validacion", default=0), 0)),
        "Muestras_Prueba": int(valor_numero(obtener_parametro(m15, "Muestras Prueba", default=0), 0)),
        "Estrategia_Evaluacion": str(obtener_parametro(m15, "Estrategia Evaluacion", default="Temporal por Nodo")),
        "Commit_C17": COMMIT_SHA,
        "Fecha_Commit_C17": COMMIT_FECHA
    })

catalogo_modelos = pd.DataFrame(filas_catalogo)

print()
print("=" * 90)
print("CATÁLOGO DE MODELOS")
print("=" * 90)
print()
display(catalogo_modelos)


CATÁLOGO DE MODELOS



,ID_Modelo_Gobernado,Experimento,Dominio,Tipo_Problema,Variable_Objetivo_Modelo,Variable_Objetivo_Cientifico,Artefacto_Modelo,Ventana,Horizonte,Variables_Predictoras,...,Ajuste_Scaler_X,Transformacion_y,Scaler_y,Ajuste_Scaler_y,Muestras_Train,Muestras_Validacion,Muestras_Prueba,Estrategia_Evaluacion,Commit_C17,Fecha_Commit_C17
0,Exp01@ce4ef76511f4,Exp01,Gestión Hídrica,Clasificación,nivel_de_riesgo,irca,modelo_Exp01_nivel_de_riesgo.keras,12,1,8,...,C14 - solo entrenamiento,No aplica,No aplica,No aplica,336,72,72,Reproducción independiente de C14 usando tenso...,ce4ef76511f43f48eb47c637b6ad10db29cf414d,2026-08-12T22:15:46Z
1,Exp04@ce4ef76511f4,Exp04,Gestión Hídrica,Regresión,VolumenUtilDiarioMasa,VolumenUtilDiarioMasa,modelo_Exp04_VolumenUtilDiarioMasa.keras,12,1,9,...,C14 - solo entrenamiento,StandardScaler,scaler_y.pkl,C14 - solo y_train,232,52,52,Reproducción independiente de C14 usando tenso...,ce4ef76511f43f48eb47c637b6ad10db29cf414d,2026-08-12T22:15:46Z


## **M5. Propósito, uso previsto y restricciones**

### Exp01

- **Objetivo operativo:** clasificar `nivel_de_riesgo`.
- **Objetivo científico asociado:** `irca`.
- **Uso previsto:** apoyar análisis de riesgo de calidad del agua en el contexto temporal y espacial representado por los datos.
- **Uso no previsto:** diagnóstico sanitario definitivo, sustitución de mediciones oficiales, decisiones regulatorias automáticas o extrapolación fuera del dominio de entrenamiento.

### Exp04

- **Objetivo:** estimar `VolumenUtilDiarioMasa`.
- **Uso previsto:** apoyar análisis experimental de disponibilidad hídrica.
- **Uso no previsto:** operación automática de infraestructura, garantía de disponibilidad futura o decisiones de asignación del recurso sin validación humana y evidencia operacional adicional.

En ambos casos, las salidas son **estimaciones predictivas**, no evidencia causal ni sustitutos del criterio técnico.


## **M6. Arquitectura y configuración vigente**

La configuración documentada debe corresponder a los C14 corregidos utilizados para producir los artefactos actualmente publicados.

| Elemento | Exp01 | Exp04 |
|---|---|---|
| Problema | Clasificación | Regresión |
| Objetivo modelo | `nivel_de_riesgo` | `VolumenUtilDiarioMasa` |
| Objetivo científico | `irca` | `VolumenUtilDiarioMasa` |
| Ventana / horizonte | 12 / 1 | 12 / 1 |
| Predictoras | 8 | 9 |
| Split | Temporal por Nodo | Temporal por Nodo |
| Escalamiento X | `RobustScaler`, fit solo Train | `RobustScaler`, fit solo Train |
| Escalamiento y | No aplica | `StandardScaler`, fit solo `y_train` |
| Núcleo | LSTM(32, tanh) | LSTM(32, tanh) |
| Regularización | Dropout(0.20) | Dropout(0.20) + Dropout(0.10) |
| Cabeza | Dense(1, sigmoid) | Dense(16, relu) → Dense(1, linear) |
| Loss | Binary crossentropy | Huber |
| Adam learning rate | 0.001 | 0.0005 |
| Batch | 32 | 32 |
| Épocas máximas | 100 | 100 |
| Early stopping | paciencia 12 | paciencia 12 |

Una modificación de datos, variables, ventana, horizonte, transformación, arquitectura, loss o procedimiento de evaluación debe producir una nueva versión identificable y nunca reemplazar silenciosamente la evidencia anterior.


## **M7. Desempeño oficial y decisión de gobierno**

C17 utiliza **exclusivamente la partición `Prueba` de C15** para la decisión técnica. Las métricas globales se conservan como descriptivas.

La decisión de gobierno es deliberadamente más conservadora que el simple estado técnico:

- una evaluación con una sola clase queda **bloqueada para uso operacional**;
- un modelo que `REQUIERE AJUSTES` queda **bloqueado**;
- un modelo `ACEPTABLE` puede aspirar a revisión/piloto, no a despliegue automático;
- un modelo `APROBADO` solo puede avanzar a **piloto controlado**, sujeto a autorización humana, trazabilidad completa y monitoreo.


In [13]:
#==========================================================================================
# SCRIPT 95
# MÉTRICAS OFICIALES Y DECISIÓN DE GOBIERNO
#==========================================================================================

filas_desempeno = []
filas_decision = []

for exp in EXPERIMENTOS:
    t = FUENTES[exp]["tablas"]
    m15 = t["c15_metadata"]
    metricas = t["c15_metricas"].copy()

    tipo = str(obtener_parametro(m15, "Tipo Problema", default=""))
    estado_c14 = str(obtener_parametro(m15, "Estado C14", default="NO DISPONIBLE"))
    estado_c15 = str(obtener_parametro(m15, "Estado C15", default="NO DISPONIBLE"))
    estado_c16 = str(obtener_parametro(
        t["c16_metadata"],
        "Estado Interpretacion C16",
        default="NO PUBLICADO / NO DISPONIBLE"
    ))

    prueba = metricas.loc[
        metricas["Particion"].astype(str).map(normalizar_texto) == "prueba"
    ]
    if len(prueba) == 0:
        raise RuntimeError(f"{exp}: C15 no contiene métricas de la partición Prueba.")

    fila = prueba.iloc[0].to_dict()
    registro_metrica = {
        "Experimento": exp,
        "Tipo_Problema": tipo,
        "N_Prueba": int(valor_numero(fila.get("N", 0), 0)),
        "Estado_C14": estado_c14,
        "Estado_C15": estado_c15,
        "Estado_C16": estado_c16
    }

    if normalizar_texto(tipo).startswith("clas"):
        for c in [
            "Accuracy", "Precision", "Recall", "F1",
            "Especificidad", "Accuracy_Balanceada", "AUC_ROC", "Clases_Presentes"
        ]:
            registro_metrica[c] = valor_numero(fila.get(c, np.nan))
    else:
        for c in ["MAE", "RMSE", "NMAE_%", "NRMSE_%", "MAPE_%", "R2"]:
            registro_metrica[c] = valor_numero(fila.get(c, np.nan))

        if t["c15_baselines"] is not None and len(t["c15_baselines"]):
            for _, b in t["c15_baselines"].iterrows():
                nombre = str(b.get("Baseline", b.get("Indicador", "")))
                registro_metrica[f"Baseline::{nombre}"] = b.get("R2", b.get("Valor", np.nan))

    filas_desempeno.append(registro_metrica)

    integridad_exp = validacion_trazabilidad.loc[
        validacion_trazabilidad["Experimento"] == exp,
        "Estado"
    ].all()

    estado_norm = normalizar_texto(estado_c15)

    if not integridad_exp:
        decision = "BLOQUEADO - INCONSISTENCIA DE TRAZABILIDAD"
        ciclo = "BLOQUEADO"
        razon = "No existe trazabilidad integral entre los artefactos gobernados."
    elif "evaluacion limitada" in estado_norm:
        decision = "NO APROBADO PARA USO OPERACIONAL - EVALUACION LIMITADA"
        ciclo = "VALIDADO CON LIMITACIONES"
        razon = "La partición oficial de prueba no permite una evaluación discriminativa completa."
    elif "requiere ajustes" in estado_norm:
        decision = "NO APROBADO PARA USO OPERACIONAL - REQUIERE AJUSTES"
        ciclo = "EN AJUSTE / INVESTIGACION"
        razon = "El desempeño oficial no cumple los criterios técnicos de aceptación del framework."
    elif "aceptable" in estado_norm:
        decision = "REVISION PARA PILOTO CONTROLADO - APROBACION CONDICIONADA"
        ciclo = "CANDIDATO A PILOTO"
        razon = "El modelo es técnicamente aceptable, pero requiere revisión humana y controles operacionales."
    elif "aprobado" in estado_norm:
        decision = "CANDIDATO A PILOTO CONTROLADO - REQUIERE AUTORIZACION HUMANA"
        ciclo = "CANDIDATO A PILOTO"
        razon = "La aprobación técnica no equivale a autorización automática de despliegue."
    else:
        decision = "REVISION MANUAL REQUERIDA"
        ciclo = "EN REVISION"
        razon = "El estado técnico no coincide con una categoría gobernada conocida."

    filas_decision.append({
        "Experimento": exp,
        "Estado_C14": estado_c14,
        "Estado_C15": estado_c15,
        "Estado_C16": estado_c16,
        "Decision_Gobierno_C17": decision,
        "Estado_Ciclo_Vida": ciclo,
        "Razon_Principal": razon,
        "Uso_Operacional_Autorizado": False,
        "Aprobacion_Humana_Requerida": True
    })

desempeno_oficial = pd.DataFrame(filas_desempeno)
decision_gobierno = pd.DataFrame(filas_decision)

print()
print("=" * 90)
print("DESEMPEÑO OFICIAL - PRUEBA")
print("=" * 90)
print()
display(desempeno_oficial)

print()
print("=" * 90)
print("DECISIÓN DE GOBIERNO C17")
print("=" * 90)
print()
display(decision_gobierno)



DESEMPEÑO OFICIAL - PRUEBA



,Experimento,Tipo_Problema,N_Prueba,Estado_C14,Estado_C15,Estado_C16,Accuracy,Precision,Recall,F1,...,Clases_Presentes,MAE,RMSE,NMAE_%,NRMSE_%,MAPE_%,R2,Baseline::R2 Baseline Media Train,Baseline::R2 Baseline Estacional 12m,Baseline::Modelo Supera Baseline Estacional
0,Exp01,Clasificación,72,EVALUACION LIMITADA - TEST CON UNA SOLA CLASE,EVALUACION LIMITADA - TEST CON UNA SOLA CLASE,NO PUBLICADO / NO DISPONIBLE,1.0,0.0,0.0,0.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Exp04,Regresión,52,MODELO REQUIERE AJUSTES,MODELO REQUIERE AJUSTES,NO PUBLICADO / NO DISPONIBLE,NaN,NaN,NaN,NaN,...,NaN,2.454124e+07,3.565002e+07,5.670764,8.237679,5.385775,-0.877819,-0.15953440827713328,-6.112020474850878,True



DECISIÓN DE GOBIERNO C17



,Experimento,Estado_C14,Estado_C15,Estado_C16,Decision_Gobierno_C17,Estado_Ciclo_Vida,Razon_Principal,Uso_Operacional_Autorizado,Aprobacion_Humana_Requerida
0,Exp01,EVALUACION LIMITADA - TEST CON UNA SOLA CLASE,EVALUACION LIMITADA - TEST CON UNA SOLA CLASE,NO PUBLICADO / NO DISPONIBLE,NO APROBADO PARA USO OPERACIONAL - EVALUACION ...,VALIDADO CON LIMITACIONES,La partición oficial de prueba no permite una ...,False,True
1,Exp04,MODELO REQUIERE AJUSTES,MODELO REQUIERE AJUSTES,NO PUBLICADO / NO DISPONIBLE,NO APROBADO PARA USO OPERACIONAL - REQUIERE AJ...,EN AJUSTE / INVESTIGACION,El desempeño oficial no cumple los criterios t...,False,True


## **M8. Registro y trazabilidad de artefactos**

Los manifiestos generados en C14 y C15 son la fuente para conservar tamaño y SHA-256 cuando estén disponibles. C17 no recalcula hashes de los binarios remotos: preserva la evidencia emitida por las etapas que los generaron y registra las URLs fuente utilizadas en esta ejecución.


In [14]:
#==========================================================================================
# SCRIPT 96
# INVENTARIO DE ARTEFACTOS Y HASHES DISPONIBLES
#==========================================================================================

filas_artefactos = []

for exp in EXPERIMENTOS:
    f = FUENTES[exp]
    t = f["tablas"]

    # Fuentes cargadas directamente por C17
    for nombre, url in f["rutas"].items():
        tabla = t[nombre]
        filas_artefactos.append({
            "Experimento": exp,
            "Etapa": nombre.split("_")[0].upper(),
            "Nombre_Logico": nombre,
            "Artefacto": os.path.basename(url),
            "URL_Fuente": url,
            "Disponible_en_C17": tabla is not None,
            "Bytes_Manifestados": np.nan,
            "SHA256_Manifestado": ""
        })

    # Evidencia de hashes producida por C14/C15
    for etapa, clave in [("C14", "c14_manifiesto"), ("C15", "c15_manifiesto")]:
        manifiesto = t.get(clave)
        if manifiesto is None or len(manifiesto) == 0:
            continue

        for _, fila in manifiesto.iterrows():
            artefacto = str(fila.get("Artefacto", fila.get("Archivo", "")))
            filas_artefactos.append({
                "Experimento": exp,
                "Etapa": etapa,
                "Nombre_Logico": "manifestado",
                "Artefacto": artefacto,
                "URL_Fuente": "",
                "Disponible_en_C17": a_bool(fila.get("Existe", True), default=True),
                "Bytes_Manifestados": fila.get("Bytes", np.nan),
                "SHA256_Manifestado": fila.get("SHA256", "")
            })

inventario_artefactos = pd.DataFrame(filas_artefactos)

print()
print("=" * 90)
print("INVENTARIO DE ARTEFACTOS")
print("=" * 90)
print()
print(f"Registros inventariados : {len(inventario_artefactos)}")
print(f"Con SHA256 manifestado  : {(inventario_artefactos['SHA256_Manifestado'].astype(str).str.len() > 0).sum()}")
print()
display(inventario_artefactos.head(25))



INVENTARIO DE ARTEFACTOS

Registros inventariados : 88
Con SHA256 manifestado  : 56



,Experimento,Etapa,Nombre_Logico,Artefacto,URL_Fuente,Disponible_en_C17,Bytes_Manifestados,SHA256_Manifestado
0,Exp01,C13,c13_registro,registro_preparacion.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
1,Exp01,C14,c14_metadata_modelo,metadata_modelo.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
2,Exp01,C14,c14_registro,registro_Exp01.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
3,Exp01,C14,c14_diagnostico,diagnostico_modelo_Exp01.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
4,Exp01,C14,c14_recomendaciones,recomendaciones_Exp01.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
5,Exp01,C14,c14_manifiesto,manifiesto_artefactos_Exp01.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
6,Exp01,C15,c15_metadata,metadata_prediccion.xlsx,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
7,Exp01,C15,c15_metricas,metricas_particiones.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
8,Exp01,C15,c15_metricas_nodo,metricas_por_nodo.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,
9,Exp01,C15,c15_integridad,validacion_integridad.csv,https://raw.githubusercontent.com/jriatiga/FRA...,True,NaN,


## **M9. Riesgos, limitaciones y uso seguro**

C17 distingue entre **riesgo metodológico**, **riesgo de datos**, **riesgo operacional** y **riesgo de interpretación**. Los controles no eliminan el riesgo; registran qué evidencia existe y qué acción debe ejecutarse antes de ampliar el uso del modelo.


In [15]:
#==========================================================================================
# SCRIPT 97
# MATRIZ DE RIESGOS POR MODELO
#==========================================================================================

filas_riesgos = []

for _, d in decision_gobierno.iterrows():
    exp = d["Experimento"]
    estado = normalizar_texto(d["Estado_C15"])

    riesgos_base = [
        (
            "Fuga de información",
            "Uso de información futura o ajuste de transformadores fuera de Train.",
            "Split temporal por Nodo; scaler X ajustado solo en Train; scaler y solo y_train en regresión.",
            "CONTROL IMPLEMENTADO"
        ),
        (
            "Desalineación de artefactos",
            "Modelo, scaler, tensores o metadata pertenecen a ejecuciones diferentes.",
            "Validación C14↔C15, manifiestos y hashes SHA-256.",
            "CONTROL IMPLEMENTADO"
        ),
        (
            "Drift de datos",
            "La distribución futura puede diferir de la histórica.",
            "Monitoreo de entradas y revisión cuando exista evidencia de cambio relevante.",
            "REQUIERE MONITOREO OPERACIONAL"
        ),
        (
            "Datos fuera de dominio",
            "Entradas fuera de rangos o estructura temporal observada.",
            "Validar esquema, faltantes, rangos y ventana antes de inferencia.",
            "REQUIERE CONTROL EN APLICACION"
        ),
        (
            "Interpretación causal indebida",
            "Confundir asociación predictiva con causalidad.",
            "C16/C17 declaran explícitamente que el framework no estima efectos causales.",
            "CONTROL DOCUMENTAL"
        ),
        (
            "Uso no previsto",
            "Utilizar el modelo para decisiones regulatorias o automáticas fuera de su propósito.",
            "Model Card, restricciones de uso y autorización humana.",
            "CONTROL DE GOBIERNO"
        )
    ]

    if "evaluacion limitada" in estado:
        riesgos_base.append((
            "Evidencia insuficiente de generalización",
            "La prueba contiene una sola clase y no permite valorar discriminación completa.",
            "Mantener bloqueado para uso operacional y complementar con validación temporal con soporte de ambas clases.",
            "ALTO / ABIERTO"
        ))

    if "requiere ajustes" in estado:
        riesgos_base.append((
            "Desempeño predictivo insuficiente",
            "Las métricas de prueba no cumplen los criterios técnicos del framework.",
            "No promover; conservar como evidencia experimental y evaluar una nueva versión si se justifica.",
            "ALTO / ABIERTO"
        ))

    for riesgo, descripcion, control, estado_riesgo in riesgos_base:
        filas_riesgos.append({
            "Experimento": exp,
            "Riesgo": riesgo,
            "Descripcion": descripcion,
            "Control_Gobierno": control,
            "Estado_Riesgo": estado_riesgo
        })

matriz_riesgos = pd.DataFrame(filas_riesgos)

print()
print("=" * 90)
print("MATRIZ DE RIESGOS")
print("=" * 90)
print()
display(matriz_riesgos)



MATRIZ DE RIESGOS



,Experimento,Riesgo,Descripcion,Control_Gobierno,Estado_Riesgo
0,Exp01,Fuga de información,Uso de información futura o ajuste de transfor...,Split temporal por Nodo; scaler X ajustado sol...,CONTROL IMPLEMENTADO
1,Exp01,Desalineación de artefactos,"Modelo, scaler, tensores o metadata pertenecen...","Validación C14↔C15, manifiestos y hashes SHA-256.",CONTROL IMPLEMENTADO
2,Exp01,Drift de datos,La distribución futura puede diferir de la his...,Monitoreo de entradas y revisión cuando exista...,REQUIERE MONITOREO OPERACIONAL
3,Exp01,Datos fuera de dominio,Entradas fuera de rangos o estructura temporal...,"Validar esquema, faltantes, rangos y ventana a...",REQUIERE CONTROL EN APLICACION
4,Exp01,Interpretación causal indebida,Confundir asociación predictiva con causalidad.,C16/C17 declaran explícitamente que el framewo...,CONTROL DOCUMENTAL
5,Exp01,Uso no previsto,Utilizar el modelo para decisiones regulatoria...,"Model Card, restricciones de uso y autorizació...",CONTROL DE GOBIERNO
6,Exp01,Evidencia insuficiente de generalización,La prueba contiene una sola clase y no permite...,Mantener bloqueado para uso operacional y comp...,ALTO / ABIERTO
7,Exp04,Fuga de información,Uso de información futura o ajuste de transfor...,Split temporal por Nodo; scaler X ajustado sol...,CONTROL IMPLEMENTADO
8,Exp04,Desalineación de artefactos,"Modelo, scaler, tensores o metadata pertenecen...","Validación C14↔C15, manifiestos y hashes SHA-256.",CONTROL IMPLEMENTADO
9,Exp04,Drift de datos,La distribución futura puede diferir de la his...,Monitoreo de entradas y revisión cuando exista...,REQUIERE MONITOREO OPERACIONAL


## **M10. Criterios de aceptación y compuertas de promoción**

Los criterios de C14/C15 se conservan como criterios técnicos. C17 agrega compuertas de gobierno:

1. trazabilidad integral;
2. integridad de artefactos;
3. evaluación temporal válida;
4. ausencia de limitación crítica en prueba;
5. estado técnico compatible con promoción;
6. propósito y restricciones documentados;
7. autorización humana;
8. plan de monitoreo antes de cualquier piloto.

**Ningún modelo queda autorizado automáticamente para producción por ejecutar C17.**


In [16]:
#==========================================================================================
# SCRIPT 98
# MATRIZ DE COMPUERTAS DE PROMOCIÓN
#==========================================================================================

filas_compuertas = []

for exp in EXPERIMENTOS:
    dec = decision_gobierno.loc[decision_gobierno["Experimento"] == exp].iloc[0]
    tra = validacion_trazabilidad.loc[validacion_trazabilidad["Experimento"] == exp]
    metricas_exp = desempeno_oficial.loc[desempeno_oficial["Experimento"] == exp].iloc[0]

    tipo = normalizar_texto(metricas_exp["Tipo_Problema"])

    controles = {
        "Trazabilidad_integral": bool(tra["Estado"].all()),
        "Estado_C14_C15_consistente": normalizar_texto(dec["Estado_C14"]) == normalizar_texto(dec["Estado_C15"]),
        "Particion_prueba_disponible": int(metricas_exp["N_Prueba"]) > 0,
        "Sin_limitacion_critica_prueba": "evaluacion limitada" not in normalizar_texto(dec["Estado_C15"]),
        "Estado_tecnico_promovible": any(
            x in normalizar_texto(dec["Estado_C15"])
            for x in ["modelo aprobado", "modelo aceptable"]
        ),
        "Aprobacion_humana": False,
        "Plan_monitoreo_operacional_activo": False
    }

    if tipo.startswith("clas"):
        controles["Prueba_con_dos_o_mas_clases"] = valor_numero(
            metricas_exp.get("Clases_Presentes", np.nan)
        ) >= 2
    else:
        controles["R2_no_negativo"] = valor_numero(metricas_exp.get("R2", np.nan)) >= 0

    for control, estado in controles.items():
        filas_compuertas.append({
            "Experimento": exp,
            "Compuerta": control,
            "Cumple": bool(estado),
            "Bloquea_Produccion": True
        })

compuertas_promocion = pd.DataFrame(filas_compuertas)

print()
print("=" * 90)
print("COMPUERTAS DE PROMOCIÓN")
print("=" * 90)
print()
display(compuertas_promocion)



COMPUERTAS DE PROMOCIÓN



,Experimento,Compuerta,Cumple,Bloquea_Produccion
0,Exp01,Trazabilidad_integral,True,True
1,Exp01,Estado_C14_C15_consistente,True,True
2,Exp01,Particion_prueba_disponible,True,True
3,Exp01,Sin_limitacion_critica_prueba,False,True
4,Exp01,Estado_tecnico_promovible,False,True
5,Exp01,Aprobacion_humana,False,True
6,Exp01,Plan_monitoreo_operacional_activo,False,True
7,Exp01,Prueba_con_dos_o_mas_clases,False,True
8,Exp04,Trazabilidad_integral,True,True
9,Exp04,Estado_C14_C15_consistente,True,True


## **M11. Monitoreo, reevaluación y reentrenamiento**

El reentrenamiento no debe dispararse únicamente por calendario. Debe existir evidencia de degradación, drift, cambio de fuente, cambio de objetivo o modificación metodológica. Para modelos no aprobados, el monitoreo operacional permanece **no activo** hasta que exista un piloto autorizado.


In [17]:
#==========================================================================================
# SCRIPT 99
# PLAN DE MONITOREO Y REENTRENAMIENTO
#==========================================================================================

plan_monitoreo = pd.DataFrame([
    {
        "Control": "Disponibilidad de variables",
        "Frecuencia": "Cada inferencia",
        "Metrica_Señal": "Variables faltantes / esquema incompatible",
        "Accion": "Bloquear inferencia y registrar incidente"
    },
    {
        "Control": "Rangos de entrada",
        "Frecuencia": "Cada inferencia / lote",
        "Metrica_Señal": "Valores fuera del dominio histórico",
        "Accion": "Advertir, registrar y revisar antes de usar la predicción"
    },
    {
        "Control": "Drift de entrada",
        "Frecuencia": "Periódica cuando exista despliegue",
        "Metrica_Señal": "Cambio relevante frente a distribución de entrenamiento",
        "Accion": "Abrir evaluación; no reentrenar automáticamente"
    },
    {
        "Control": "Desempeño observado",
        "Frecuencia": "Cuando se disponga del valor real",
        "Metrica_Señal": "Deterioro sostenido frente a referencia validada",
        "Accion": "Reevaluar versión y decidir si procede reentrenamiento"
    },
    {
        "Control": "Cambio de fuente o definición",
        "Frecuencia": "Ante cambio",
        "Metrica_Señal": "Nueva fuente, unidad, objetivo, variable o proceso",
        "Accion": "Crear nueva versión; no sobrescribir la versión gobernada"
    },
    {
        "Control": "Integridad de artefactos",
        "Frecuencia": "Antes de despliegue / auditoría",
        "Metrica_Señal": "Hash o versión diferente",
        "Accion": "Bloquear y reconciliar artefactos"
    }
])

criterios_reentrenamiento = pd.DataFrame({
    "Evento": [
        "Deterioro significativo de métricas",
        "Drift material en variables de entrada",
        "Aparición de nuevos rangos o patrones",
        "Cambio de fuentes o procedimiento de medición",
        "Cambio de variable objetivo",
        "Cambio de predictoras",
        "Cambio de ventana u horizonte",
        "Cambio de arquitectura / loss / hiperparámetros"
    ],
    "Accion_Gobierno": [
        "Reevaluar y documentar evidencia",
        "Reevaluar y documentar evidencia",
        "Revisar dominio de validez",
        "Crear nueva versión",
        "Crear nuevo experimento/versión",
        "Crear nueva versión",
        "Crear nueva versión",
        "Crear nueva versión"
    ]
})

print()
print("=" * 90)
print("PLAN DE MONITOREO")
print("=" * 90)
print()
display(plan_monitoreo)
print()
display(criterios_reentrenamiento)



PLAN DE MONITOREO



,Control,Frecuencia,Metrica_Señal,Accion
0,Disponibilidad de variables,Cada inferencia,Variables faltantes / esquema incompatible,Bloquear inferencia y registrar incidente
1,Rangos de entrada,Cada inferencia / lote,Valores fuera del dominio histórico,"Advertir, registrar y revisar antes de usar la..."
2,Drift de entrada,Periódica cuando exista despliegue,Cambio relevante frente a distribución de entr...,Abrir evaluación; no reentrenar automáticamente
3,Desempeño observado,Cuando se disponga del valor real,Deterioro sostenido frente a referencia validada,Reevaluar versión y decidir si procede reentre...
4,Cambio de fuente o definición,Ante cambio,"Nueva fuente, unidad, objetivo, variable o pro...",Crear nueva versión; no sobrescribir la versió...
5,Integridad de artefactos,Antes de despliegue / auditoría,Hash o versión diferente,Bloquear y reconciliar artefactos


,Evento,Accion_Gobierno
0,Deterioro significativo de métricas,Reevaluar y documentar evidencia
1,Drift material en variables de entrada,Reevaluar y documentar evidencia
2,Aparición de nuevos rangos o patrones,Revisar dominio de validez
3,Cambio de fuentes o procedimiento de medición,Crear nueva versión
4,Cambio de variable objetivo,Crear nuevo experimento/versión
5,Cambio de predictoras,Crear nueva versión
6,Cambio de ventana u horizonte,Crear nueva versión
7,Cambio de arquitectura / loss / hiperparámetros,Crear nueva versión


## **M12. Trazabilidad del ciclo de vida**

La secuencia gobernada vigente es:

**C13 preparación → C14 entrenamiento → C15 evaluación → C16 interpretación → C17 gobierno → piloto/aplicación solo si las compuertas lo permiten.**

C17 no altera los artefactos históricos. Una nueva configuración debe conservar el modelo anterior y generar una nueva identificación/versionamiento.


In [18]:
#==========================================================================================
# SCRIPT 100
# MAPA DE TRAZABILIDAD DEL CICLO DE VIDA
#==========================================================================================

trazabilidad_ciclo = pd.DataFrame([
    ["C13", "Preparación ML", "registro_preparacion.csv", "Predictoras, objetivo, ventana, secuencias y decisión de no escalar en C13"],
    ["C14", "Modelado", "registro_<Exp>.csv / metadata_modelo.csv", "Split temporal, scalers ajustados en train, arquitectura, entrenamiento, diagnóstico y modelo"],
    ["C15", "Evaluación", "metadata_prediccion.csv / metricas_particiones.csv", "Reproducción de inferencia, métricas oficiales de prueba e integridad"],
    ["C16", "Interpretación", "metadata_interpretacion.csv", "Interpretación predictiva, limitaciones, hallazgos y recomendaciones"],
    ["C17", "Gobierno", "registro_gobierno_modelos.csv", "Decisión de uso, riesgos, versionamiento, monitoreo y Model Card"]
], columns=["Etapa", "Funcion", "Evidencia_Principal", "Control_Gobernado"])

print()
print("=" * 90)
print("TRAZABILIDAD DEL CICLO DE VIDA")
print("=" * 90)
print()
display(trazabilidad_ciclo)



TRAZABILIDAD DEL CICLO DE VIDA



,Etapa,Funcion,Evidencia_Principal,Control_Gobernado
0,C13,Preparación ML,registro_preparacion.csv,"Predictoras, objetivo, ventana, secuencias y d..."
1,C14,Modelado,registro_<Exp>.csv / metadata_modelo.csv,"Split temporal, scalers ajustados en train, ar..."
2,C15,Evaluación,metadata_prediccion.csv / metricas_particiones...,"Reproducción de inferencia, métricas oficiales..."
3,C16,Interpretación,metadata_interpretacion.csv,"Interpretación predictiva, limitaciones, halla..."
4,C17,Gobierno,registro_gobierno_modelos.csv,"Decisión de uso, riesgos, versionamiento, moni..."


## **M13. Model Cards automáticas**

Se genera una ficha Markdown por experimento. La ficha separa propósito, configuración, desempeño, decisión de gobierno, limitaciones y uso no previsto. Su contenido se deriva de los artefactos cargados, no de valores escritos manualmente.


In [19]:
#==========================================================================================
# SCRIPT 101
# GENERACIÓN DE MODEL CARDS
#==========================================================================================

rutas_model_cards = []

for exp in EXPERIMENTOS:
    cat = catalogo_modelos.loc[catalogo_modelos["Experimento"] == exp].iloc[0]
    dec = decision_gobierno.loc[decision_gobierno["Experimento"] == exp].iloc[0]
    met = desempeno_oficial.loc[desempeno_oficial["Experimento"] == exp].iloc[0]

    tipo_norm = normalizar_texto(cat["Tipo_Problema"])

    if tipo_norm.startswith("clas"):
        metricas_txt = (
            f"- Accuracy: {met.get('Accuracy', np.nan)}\n"
            f"- Precision: {met.get('Precision', np.nan)}\n"
            f"- Recall: {met.get('Recall', np.nan)}\n"
            f"- F1: {met.get('F1', np.nan)}\n"
            f"- Accuracy balanceada: {met.get('Accuracy_Balanceada', np.nan)}\n"
            f"- AUC ROC: {met.get('AUC_ROC', np.nan)}\n"
            f"- Clases presentes en prueba: {met.get('Clases_Presentes', np.nan)}"
        )
        uso = "Apoyo experimental al análisis de nivel de riesgo asociado al IRCA."
    else:
        metricas_txt = (
            f"- MAE: {met.get('MAE', np.nan)}\n"
            f"- RMSE: {met.get('RMSE', np.nan)}\n"
            f"- NMAE %: {met.get('NMAE_%', np.nan)}\n"
            f"- NRMSE %: {met.get('NRMSE_%', np.nan)}\n"
            f"- MAPE %: {met.get('MAPE_%', np.nan)}\n"
            f"- R²: {met.get('R2', np.nan)}"
        )
        uso = "Apoyo experimental al análisis predictivo de VolumenUtilDiarioMasa."

    texto = f"""# Model Card — {exp}

## Identificación
- ID gobernado: {cat['ID_Modelo_Gobernado']}
- Dominio: {cat['Dominio']}
- Tipo de problema: {cat['Tipo_Problema']}
- Objetivo del modelo: {cat['Variable_Objetivo_Modelo']}
- Objetivo científico: {cat['Variable_Objetivo_Cientifico']}
- Artefacto del modelo: {cat['Artefacto_Modelo']}

## Propósito
{uso}

## Configuración
- Ventana: {cat['Ventana']}
- Horizonte: {cat['Horizonte']}
- Predictoras: {cat['Variables_Predictoras']}
- Variables: {cat['Lista_Predictoras']}
- Transformación X: {cat['Transformacion_X']} — {cat['Ajuste_Scaler_X']}
- Transformación y: {cat['Transformacion_y']} — {cat['Ajuste_Scaler_y']}
- Split: {cat['Estrategia_Evaluacion']}

## Desempeño oficial de prueba
{metricas_txt}

## Estado
- Estado C14: {dec['Estado_C14']}
- Estado C15: {dec['Estado_C15']}
- Estado C16: {dec['Estado_C16']}
- Decisión C17: {dec['Decision_Gobierno_C17']}
- Uso operacional autorizado: {dec['Uso_Operacional_Autorizado']}

## Limitaciones y uso no previsto
- No sustituye mediciones oficiales ni criterio profesional.
- No autoriza decisiones regulatorias automáticas.
- No debe extrapolarse fuera del dominio de datos sin validación.
- No implica causalidad.
- Cualquier cambio metodológico requiere nueva versión.

## Trazabilidad
- Commit de referencia C17: {COMMIT_SHA}
- Fecha commit: {COMMIT_FECHA}
- Generado: {datetime.now().isoformat(timespec='seconds')}
"""

    ruta = os.path.join(CARPETA_SALIDA, f"Model_Card_{exp}.md")
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(texto)
    rutas_model_cards.append(ruta)

print("Model Cards generadas:")
for ruta in rutas_model_cards:
    print("-", ruta)


Model Cards generadas:
- gobierno_modelos/Model_Card_Exp01.md
- gobierno_modelos/Model_Card_Exp04.md


## **M14. Exportación del Registro de Gobierno**

C17 exporta las evidencias de gobierno en CSV/XLSX y crea un manifiesto SHA-256 de sus propios artefactos. El paquete ZIP facilita su publicación y auditoría.


In [20]:
#==========================================================================================
# SCRIPT 102
# EXPORTACIÓN DE ARTEFACTOS DE GOBIERNO
#==========================================================================================

salidas = {
    "catalogo_modelos": catalogo_modelos,
    "decision_gobierno": decision_gobierno,
    "desempeno_oficial": desempeno_oficial,
    "validacion_trazabilidad": validacion_trazabilidad,
    "inventario_artefactos": inventario_artefactos,
    "matriz_riesgos": matriz_riesgos,
    "compuertas_promocion": compuertas_promocion,
    "plan_monitoreo": plan_monitoreo,
    "criterios_reentrenamiento": criterios_reentrenamiento,
    "trazabilidad_ciclo_vida": trazabilidad_ciclo
}

rutas_generadas = []

for nombre, df in salidas.items():
    ruta_csv = os.path.join(CARPETA_SALIDA, f"{nombre}.csv")
    ruta_xlsx = os.path.join(CARPETA_SALIDA, f"{nombre}.xlsx")
    df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
    df.to_excel(ruta_xlsx, index=False)
    rutas_generadas.extend([ruta_csv, ruta_xlsx])

rutas_generadas.extend(rutas_model_cards)

metadata_gobierno = pd.DataFrame({
    "Parametro": [
        "Framework",
        "Modulo",
        "Experimentos Gobernados",
        "Repositorio",
        "Rama",
        "Commit Referencia",
        "Fecha Commit",
        "Modelos con Uso Operacional Autorizado",
        "C16 Usado si Disponible",
        "Fecha Ejecucion C17"
    ],
    "Valor": [
        "Framework V7",
        "C17 - Gobierno de Modelos",
        ";".join(EXPERIMENTOS),
        REPO,
        RAMA,
        COMMIT_SHA,
        COMMIT_FECHA,
        int(decision_gobierno["Uso_Operacional_Autorizado"].sum()),
        USAR_C16_SI_DISPONIBLE,
        datetime.now().isoformat(timespec="seconds")
    ]
})

ruta_metadata_csv = os.path.join(CARPETA_SALIDA, "metadata_gobierno.csv")
ruta_metadata_xlsx = os.path.join(CARPETA_SALIDA, "metadata_gobierno.xlsx")
metadata_gobierno.to_csv(ruta_metadata_csv, index=False, encoding="utf-8-sig")
metadata_gobierno.to_excel(ruta_metadata_xlsx, index=False)
rutas_generadas.extend([ruta_metadata_csv, ruta_metadata_xlsx])

# Manifiesto propio de C17
filas_manifiesto_c17 = []
for ruta in rutas_generadas:
    filas_manifiesto_c17.append({
        "Artefacto": ruta,
        "Existe": os.path.exists(ruta),
        "Bytes": os.path.getsize(ruta) if os.path.exists(ruta) else np.nan,
        "SHA256": sha256_archivo(ruta) if os.path.exists(ruta) else ""
    })

manifiesto_c17 = pd.DataFrame(filas_manifiesto_c17)
ruta_manifiesto = os.path.join(CARPETA_SALIDA, "manifiesto_artefactos_C17.csv")
manifiesto_c17.to_csv(ruta_manifiesto, index=False, encoding="utf-8-sig")

# ZIP final
ruta_zip = "C17_Gobierno_Modelos.zip"
with zipfile.ZipFile(ruta_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for ruta in rutas_generadas + [ruta_manifiesto]:
        zf.write(ruta, arcname=os.path.relpath(ruta, CARPETA_SALIDA))

print()
print("=" * 90)
print("ARTEFACTOS C17 EXPORTADOS")
print("=" * 90)
print()
print(f"Carpeta : {CARPETA_SALIDA}")
print(f"ZIP     : {ruta_zip}")
print()
display(manifiesto_c17)



ARTEFACTOS C17 EXPORTADOS

Carpeta : gobierno_modelos
ZIP     : C17_Gobierno_Modelos.zip



,Artefacto,Existe,Bytes,SHA256
0,gobierno_modelos/catalogo_modelos.csv,True,1432,325908f7bb9e723a0a63e373027dca540ff2bb9845e760...
1,gobierno_modelos/catalogo_modelos.xlsx,True,5778,9f920dd895516f84269225170e97ed033aad8d97d11fd7...
2,gobierno_modelos/decision_gobierno.csv,True,720,40b5ca4337917894eda92cd7075a6c1d0d6182963bcd1c...
3,gobierno_modelos/decision_gobierno.xlsx,True,5391,49ef0c28b56f6655c745e02b5c164411e094fdd33d45fc...
4,gobierno_modelos/desempeno_oficial.csv,True,752,f3ad19d36733028571dadbaab4a88036df9d534d99682d...
5,gobierno_modelos/desempeno_oficial.xlsx,True,5584,a8681290fedcae98e4e965121ba9fc27333c290001735f...
6,gobierno_modelos/validacion_trazabilidad.csv,True,694,7f178e06c61528334377c92457d8520e295f76ad849b98...
7,gobierno_modelos/validacion_trazabilidad.xlsx,True,5341,6436be85cb58e79fb49d709addd2ed855b3d274cbc7a18...
8,gobierno_modelos/inventario_artefactos.csv,True,13732,447292d518083c9245931653a46239bdcb459954661546...
9,gobierno_modelos/inventario_artefactos.xlsx,True,11321,afa8a5774636863c587468e5e9210187465a361063e17d...


## **M15. Validación final del Gobierno**

La validación final comprueba que C17 generó las evidencias mínimas de gobierno y que cada experimento tiene una decisión explícita. Una decisión de bloqueo o no aprobación **no es un fallo de C17**: es una salida válida del proceso de gobierno.


In [21]:
#==========================================================================================
# SCRIPT 103
# VALIDACIÓN FINAL C17
#==========================================================================================

controles_c17 = {
    "Experimentos_completos": set(catalogo_modelos["Experimento"]) == set(EXPERIMENTOS),
    "Decisiones_completas": set(decision_gobierno["Experimento"]) == set(EXPERIMENTOS),
    "Metricas_prueba_disponibles": set(desempeno_oficial["Experimento"]) == set(EXPERIMENTOS),
    "Trazabilidad_registrada": len(validacion_trazabilidad) > 0,
    "Riesgos_registrados": len(matriz_riesgos) > 0,
    "Compuertas_registradas": len(compuertas_promocion) > 0,
    "Plan_monitoreo_registrado": len(plan_monitoreo) > 0,
    "Model_Cards_generadas": all(os.path.exists(r) for r in rutas_model_cards),
    "Manifesto_C17_generado": os.path.exists(ruta_manifiesto),
    "ZIP_generado": os.path.exists(ruta_zip),
    "Ningun_despliegue_automatico": not decision_gobierno["Uso_Operacional_Autorizado"].any()
}

validacion_final_c17 = pd.DataFrame({
    "Control": list(controles_c17.keys()),
    "Estado": list(controles_c17.values())
})

ruta_validacion_final = os.path.join(CARPETA_SALIDA, "validacion_final_C17.csv")
validacion_final_c17.to_csv(ruta_validacion_final, index=False, encoding="utf-8-sig")

print()
print("=" * 90)
print("VALIDACIÓN FINAL C17")
print("=" * 90)
print()

for control, estado in controles_c17.items():
    print(f"- {control:<42}: {estado}")

print()

if not all(controles_c17.values()):
    raise RuntimeError("C17 detectó inconsistencias en sus artefactos de gobierno.")

print("Resultado : GOBIERNO DE MODELOS GENERADO Y VALIDADO")
print()
print("Decisiones vigentes:")
for _, fila in decision_gobierno.iterrows():
    print(f"- {fila['Experimento']}: {fila['Decision_Gobierno_C17']}")
print()
print("IMPORTANTE: C17 documenta y controla el uso; no autoriza despliegue automático.")



VALIDACIÓN FINAL C17

- Experimentos_completos                    : True
- Decisiones_completas                      : True
- Metricas_prueba_disponibles               : True
- Trazabilidad_registrada                   : True
- Riesgos_registrados                       : True
- Compuertas_registradas                    : True
- Plan_monitoreo_registrado                 : True
- Model_Cards_generadas                     : True
- Manifesto_C17_generado                    : True
- ZIP_generado                              : True
- Ningun_despliegue_automatico              : True

Resultado : GOBIERNO DE MODELOS GENERADO Y VALIDADO

Decisiones vigentes:
- Exp01: NO APROBADO PARA USO OPERACIONAL - EVALUACION LIMITADA
- Exp04: NO APROBADO PARA USO OPERACIONAL - REQUIERE AJUSTES

IMPORTANTE: C17 documenta y controla el uso; no autoriza despliegue automático.


## **17.16 Conclusión de Gobierno**

El Gobierno de Modelos de Framework V7 queda estructurado sobre evidencia reproducible y no sobre la mera existencia de un archivo `.keras`.

La promoción de cualquier modelo requiere coherencia entre preparación, entrenamiento, evaluación e interpretación; control de versiones; evidencia de desempeño; restricciones de uso; monitoreo y autorización humana. Un modelo puede formar parte del registro histórico aun cuando su decisión sea **no aprobado para uso operacional**.
